# exp069_pixiux_pf_beam_direct_submit_audit inference

PF/Beam direct submission audit for the Pixiux public replay surface. This notebook generates test-side public replay tracker features, writes raw `likpf_mean` TVT predictions to `submission.csv`, and saves alternative direct candidates for diagnostics. It does not load or use LightGBM boosters.

## Contents

1. Setup and configuration
2. Raw competition input check
3. Direct PF/Beam submission audit
4. Submission and artifacts

## 1. Setup and configuration

In [ ]:
from pathlib import Path
import json
import pandas as pd

from settings import ExperimentPaths, load_config, get_nested
from public_notebook_replay_audit import run_pixiux_pf_beam_direct_submit_audit

def cfg_get(config, dotted_key, default=None):
    value = get_nested(config, dotted_key)
    return default if value is None else value

paths = ExperimentPaths()
paths.ensure_output_dirs()
config = load_config()

print("Experiment:", config["experiment"]["name"])
print("Route:", config["experiment"]["route"])
print("Mode:", cfg_get(config, "inference.mode", "pixiux_pf_beam_direct_submit_audit"))
print("Candidate:", cfg_get(config, "inference.candidate", "likpf_mean"))
print("Submission path:", paths.submission_path)

## 2. Raw competition input check

In [ ]:
train_dir = paths.train_data_dir
test_dir = paths.test_data_dir
sample_path = paths.sample_submission_path
train_files = sorted(train_dir.glob("*__horizontal_well.csv"))
test_files = sorted(test_dir.glob("*__horizontal_well.csv"))
print("Train dir:", train_dir, "wells=", len(train_files))
print("Test dir:", test_dir, "wells=", len(test_files))
print("Sample submission:", sample_path, "exists=", sample_path.exists())
if not train_files:
    raise FileNotFoundError(f"No train wells found under {train_dir}")
if not test_files:
    raise FileNotFoundError(f"No test wells found under {test_dir}")
if not sample_path.exists():
    raise FileNotFoundError(f"Sample submission not found: {sample_path}")

## 3. Direct PF/Beam submission audit

In [ ]:
summary = run_pixiux_pf_beam_direct_submit_audit(
    data_dir=paths.raw_data_dir,
    output_dir=paths.artifacts_dir,
    submission_path=paths.submission_path,
    n_jobs=cfg_get(config, "runtime.num_workers", 8),
    pf_seeds=cfg_get(config, "audit.pf_seeds", 128),
    pf_particles=cfg_get(config, "audit.pf_particles", 500),
    fast=bool(cfg_get(config, "audit.fast", False)),
    use_gpu=str(cfg_get(config, "audit.use_gpu", "auto")),
    max_wells=cfg_get(config, "audit.max_wells"),
    deterministic=bool(cfg_get(config, "audit.deterministic", True)),
    candidate=cfg_get(config, "inference.candidate", "likpf_mean"),
    sample_submission_path=paths.sample_submission_path,
    submission_target_column=cfg_get(config, "data.submission_target_column", "tvt"),
    reference_submission_paths=cfg_get(config, "audit.reference_submission_paths", []),
)
print(json.dumps(summary, indent=2))

## 4. Submission and artifacts

In [ ]:
metrics = pd.read_csv(paths.artifacts_dir / "pixiux_pf_beam_direct_submission_metrics.csv")
candidate_metrics = pd.read_csv(paths.artifacts_dir / "pixiux_pf_beam_direct_candidate_metrics.csv")
comparison = pd.read_csv(paths.artifacts_dir / "pixiux_pf_beam_direct_reference_comparison.csv")
submission = pd.read_csv(paths.submission_path)

display(metrics)
display(candidate_metrics)
display(comparison)
display(submission.head())
print("submission rows:", len(submission))
print("submission columns:", list(submission.columns))
print("submission path:", paths.submission_path)
print("prediction range:", float(submission.iloc[:, 1].min()), float(submission.iloc[:, 1].max()))